# Computed columns and data types

Practice notebook for the datafromzero SQL lesson **Computed columns and data types**. The engine is DuckDB and the data is
**cafe-sales**: five tables from a fictional chain of three cafes (synthetic data, CC BY 4.0).

Run the cells from top to bottom. The worked example is written out so you can run it and read the results. The
exercises start empty or half done: write your own query there. The solutions are in the lesson, not here.

Each query sits between `con.sql("""` and `""").show()`: only the text inside the triple quotes is SQL, and `.show()`
prints the result as a table. The cells leave out the closing `;` that the lesson shows.

## Setup

Run this cell first. It builds an empty database in memory and loads the five cafe-sales tables into it.

In [ ]:
%pip install -q duckdb==2.0.0  # the version these lessons were checked with
import duckdb, urllib.request

URL = "https://raw.githubusercontent.com/datafromzero/datafromzero.github.io/v0.1.0/datasets/cafe-sales/v1/cafe-sales.sql"
script = urllib.request.urlopen(URL).read().decode("utf-8")

con = duckdb.connect()  # an empty database that lives in memory
con.execute(script)     # creates and fills the five cafe-sales tables
print("cafe-sales is loaded")

## Worked example

### Step 1. Calculate a column

In [ ]:
con.sql("""
SELECT product_name, unit_price_usd, unit_price_usd * 0.80 AS happy_price_usd
FROM products
WHERE category IN ('coffee', 'tea')
""").show()

### Step 2. Mind the order of operations

In [ ]:
con.sql("""
SELECT product_name,
       unit_price_usd * 2 - 0.50 AS pair_price_usd,
       unit_price_usd * (2 - 0.50) AS wrong_pair_usd
FROM products
WHERE product_id <= 3
""").show()

### Step 3. Read the types

In [ ]:
con.sql("""
DESCRIBE products
""").show()

In [ ]:
con.sql("""
DESCRIBE SELECT product_name, unit_price_usd, unit_price_usd * 0.80 AS happy_price_usd
FROM products
WHERE category IN ('coffee', 'tea')
""").show()

### Step 4. Cast to exact cents

In [ ]:
con.sql("""
SELECT product_name, unit_price_usd,
       CAST(unit_price_usd * 0.80 AS DECIMAL(6,2)) AS happy_price_usd
FROM products
WHERE category IN ('coffee', 'tea')
""").show()

### Step 5. See why money is not DOUBLE

In [ ]:
con.sql("""
SELECT product_name, CAST(unit_price_usd AS DOUBLE) * 0.80 AS happy_price_double
FROM products
WHERE category IN ('coffee', 'tea')
""").show()

### Step 6. Meet integer division

In [ ]:
con.sql("""
SELECT order_item_id, quantity,
       quantity / 2 AS halved,
       quantity // 2 AS pairs,
       quantity % 2 AS single
FROM order_items
WHERE order_id = 1001
""").show()

### Step 7. Cast text to a number or a date, and back

In [ ]:
con.sql("""
SELECT CAST('4.50' AS DECIMAL(6,2)) AS price,
       CAST('2026-03-31' AS DATE) AS day,
       CAST(1001 AS VARCHAR) AS order_text
""").show()

### Step 8. Verify

In [ ]:
con.sql("""
SELECT product_name
FROM products
WHERE category IN ('coffee', 'tea')
  AND unit_price_usd - CAST(unit_price_usd * 0.80 AS DECIMAL(6,2)) <> unit_price_usd * 0.20
""").show()

## Exercise 1. Happy-hour prices for the bakery items

The first lines are given. After the comma at the end of the `SELECT` line, add the third column: 20 percent off, cast to
`DECIMAL(6,2)`, named `happy_price_usd`. DuckDB accepts a trailing comma there, while other databases may not.

In [ ]:
con.sql("""
SELECT product_name, unit_price_usd,
FROM products
WHERE category = 'bakery'
""").show()

## Exercise 2. Price a pair from a text price list

A supplier sent a price list in which every column is text. Run the first cell: it creates the table `price_list_text`
(four rows) so you can query it. Then, before you write any query, write one sentence in the second cell saying what the
query will return and what type each column will have.

The pair price of an item is two items minus a 0.50 coupon. Show the item, its pair price as an exact decimal named
`pair_price_usd`, and the day the price starts as a date named `valid_from`.

In [ ]:
con.execute("""
CREATE OR REPLACE TABLE price_list_text (
  item VARCHAR,
  price_text VARCHAR,
  valid_from_text VARCHAR
);
INSERT INTO price_list_text VALUES
  ('Espresso', '3.00', '2026-07-01'),
  ('Latte', '4.50', '2026-07-01'),
  ('Chai Latte', '4.75', '2026-07-15'),
  ('Croissant', '3.25', '2026-07-15');
""")
print("price_list_text is created")

In [ ]:
# My expected result, in words:
#

In [ ]:
# Step 1: look at the types of price_list_text with DESCRIBE

In [ ]:
# Step 2: your query

## Exercise 3. Spot the flaw

A colleague wants half of each line's quantity in order 1001, with decimals such as 0.5, and runs this query. It gives a table with
no error. Compare the result with the quantities: is it really half? Then write a query that works.

In [ ]:
con.sql("""
SELECT order_item_id, quantity, quantity // 2 AS half_the_quantity
FROM order_items
WHERE order_id = 1001
""").show()

In [ ]:
# Your corrected query: